# PyTerrier dense retrieval (E5 / Qwen3 via FlexIndex)

Notebook version of `pt_dense_retrieval.py`. It reuses that module's
`DenseRetrievalConfig` and `DenseRetrievalPipeline` directly, so behaviour is
identical to the command-line script -- this notebook is just a more
interactive way to run it and inspect results per query.

**Environment note:** PyTerrier embeds a JVM inside the Python process via
`pyjnius` (even for dense/FlexIndex retrieval, since importing `pyterrier`
starts one). On this machine, the system/Homebrew Python crashes trying to
start that JVM (a JIT/code-signing restriction), while the Anaconda Python
distribution (`/opt/anaconda3/bin/python3`) does not. If you hit a JVM crash
running this notebook, pick the Anaconda Python as this notebook's kernel.

## 1. Setup

In [1]:
import sys, os

# So `import pt_dense_retrieval` finds the module in src/ regardless of the
# working directory the kernel was started from.
PROJECT_DIR = os.path.abspath(".")
SRC_DIR = os.path.join(PROJECT_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import pandas as pd
from pt_dense_retrieval import DenseRetrievalConfig, DenseRetrievalPipeline

pd.set_option("display.max_colwidth", 80)

## 2. Configure the run

Edit these to point at a different `*_chunks_docs.csv` / `*_queries_topic_names_qid_query.csv`
pair, switch retriever (`"e5"` or `"qwen3"`), or change the model checkpoint.

In [ ]:
DOCS_CSV = "data/euaa_asylum_report_chunks_docs.csv"
TOPICS_CSV = "data/euaa_asylum_report_queries_topic_names_qid_query.csv"
RETRIEVER = "e5"                     # "e5" | "qwen3"
TOP_K = 100
E5_MODEL = "base"                    # "small" | "base" | "large", or a full HF checkpoint id
QWEN3_MODEL = "Qwen/Qwen3-Embedding-0.6B"

In [ ]:
config = DenseRetrievalConfig(
    docs_csv=DOCS_CSV,
    topics_csv=TOPICS_CSV,
    retriever=RETRIEVER,
    top_k=TOP_K,
    e5_model=E5_MODEL,
    qwen3_model=QWEN3_MODEL,
)
config

## 3. Run the pipeline

`run()` builds the FlexIndex (or reuses one already built at the same
`index_dir`), encodes the queries, retrieves the top-k per query, joins in
the retrieved document's text, and writes the output CSV -- all in one call.
It also returns the resulting DataFrame.

In [3]:
pipeline = DenseRetrievalPipeline(config)
results_df = pipeline.run()

output_path = config.output_csv or pipeline._default_output_path()
print(f"Retrieved {len(results_df)} rows ({results_df['qid'].nunique()} queries) -> {output_path}")
results_df.head(10)

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/qbr926/Desktop/actor/src/pt_dense_retrieval.py:85: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


NumpyRetriever scoring:   0%|          | 0/1 [00:00<?, ?docbatch/s]

NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 354.25docbatch/s]

Retrieved 500 rows (5 queries) -> data/euaa_asylum_report_queries_topic_names_qid_query_e5.csv


,qid,query,docno,docid,score,rank,text
0,0,Asylum Definition And Assessment,923_chunk000,264,0.852975,0,An applicant must submit a request for protection with which he asserts targ...
1,0,Asylum Definition And Assessment,577_chunk000,314,0.847139,1,This case concerns the validity of asylum application in accordance with § 2...
2,0,Asylum Definition And Assessment,563_chunk000,348,0.836106,2,"The case concerns the moment at which an asylum application is lodged, in pa..."
3,0,Asylum Definition And Assessment,4705_chunk000,446,0.835089,3,Judgment concerning the freedom of movement of asylum seekers in Ceuta and M...
4,0,Asylum Definition And Assessment,643_chunk000,388,0.832142,4,As per the published press release: The Council of State validates the crite...
5,0,Asylum Definition And Assessment,978_chunk001,208,0.832120,5,The Court of Cassation is of the opinion that in the plaintiff's case defini...
6,0,Asylum Definition And Assessment,590_chunk000,350,0.830783,6,In an urgent action for the «granting of asylum or of a residence permit for...
7,0,Asylum Definition And Assessment,938_chunk000,199,0.825308,7,IThe foreign national has submitted a subsequent asylum application. The dec...
8,0,Asylum Definition And Assessment,964_chunk001,221,0.824990,8,The line of reasoning of the defendant who claims that the plaintiff had a l...
9,0,Asylum Definition And Assessment,3277_chunk001,119,0.824974,9,"When the asylum procedure starts, applicants are usually assigned a lawyer t..."


## 4. Inspect results for one query

In [4]:
sample_qid = results_df["qid"].iloc[0]
cols = ["qid", "query", "docno", "rank", "score", "text"]
results_df[results_df["qid"] == sample_qid][cols].head(10)

,qid,query,docno,rank,score,text
0,0,Asylum Definition And Assessment,923_chunk000,0,0.852975,An applicant must submit a request for protection with which he asserts targ...
1,0,Asylum Definition And Assessment,577_chunk000,1,0.847139,This case concerns the validity of asylum application in accordance with § 2...
2,0,Asylum Definition And Assessment,563_chunk000,2,0.836106,"The case concerns the moment at which an asylum application is lodged, in pa..."
3,0,Asylum Definition And Assessment,4705_chunk000,3,0.835089,Judgment concerning the freedom of movement of asylum seekers in Ceuta and M...
4,0,Asylum Definition And Assessment,643_chunk000,4,0.832142,As per the published press release: The Council of State validates the crite...
5,0,Asylum Definition And Assessment,978_chunk001,5,0.832120,The Court of Cassation is of the opinion that in the plaintiff's case defini...
6,0,Asylum Definition And Assessment,590_chunk000,6,0.830783,In an urgent action for the «granting of asylum or of a residence permit for...
7,0,Asylum Definition And Assessment,938_chunk000,7,0.825308,IThe foreign national has submitted a subsequent asylum application. The dec...
8,0,Asylum Definition And Assessment,964_chunk001,8,0.824990,The line of reasoning of the defendant who claims that the plaintiff had a l...
9,0,Asylum Definition And Assessment,3277_chunk001,9,0.824974,"When the asylum procedure starts, applicants are usually assigned a lawyer t..."


## 5. Optional: compare against the other dense encoder

Reruns retrieval with the other retriever (`e5` <-> `qwen3`) on the same
docs/topics, so you can eyeball how the two rankings differ for the sample
query above. Skip this cell if you only need one encoder.

In [5]:
other_retriever = "qwen3" if RETRIEVER == "e5" else "e5"
other_config = DenseRetrievalConfig(
    docs_csv=DOCS_CSV,
    topics_csv=TOPICS_CSV,
    retriever=other_retriever,
    top_k=TOP_K,
    e5_model=E5_MODEL,
    qwen3_model=QWEN3_MODEL,
)
other_pipeline = DenseRetrievalPipeline(other_config)
other_df = other_pipeline.run()

other_output_path = other_config.output_csv or other_pipeline._default_output_path()
print(f"[{other_retriever}] retrieved {len(other_df)} rows -> {other_output_path}")
other_df[other_df["qid"] == sample_qid][["qid", "query", "docno", "rank", "score", "text"]].head(10)

NumpyRetriever scoring:   0%|          | 0/1 [00:00<?, ?docbatch/s]

NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 375.97docbatch/s]

[qwen3] retrieved 500 rows -> data/euaa_asylum_report_queries_topic_names_qid_query_qwen3.csv


,qid,query,docno,rank,score,text
0,0,Asylum Definition And Assessment,148_chunk000,0,0.717146,The case concerns “sur place” asylum claims.
1,0,Asylum Definition And Assessment,108_chunk001,1,0.663836,The applicant subsequently lodged a request for his asylum application to be...
2,0,Asylum Definition And Assessment,4036_chunk000,2,0.650532,The case concerned an application for asylum submitted by an applicant from ...
3,0,Asylum Definition And Assessment,23_chunk000,3,0.643393,The Finnish Immigration Service had rejected A 's application for asylum and...
4,0,Asylum Definition And Assessment,4231_chunk000,4,0.641884,The case concerned the question whether the State Secretary had correctly es...
5,0,Asylum Definition And Assessment,510_chunk000,5,0.640862,The Court of Sessions found there was no unlawful discrimination arising fro...
6,0,Asylum Definition And Assessment,951_chunk000,6,0.639135,The case concerns the risk situation in case of return to the country of ori...
7,0,Asylum Definition And Assessment,4235_chunk000,7,0.638808,"The case concerned two Iranian applicant, a father and his minor child, who ..."
8,0,Asylum Definition And Assessment,1564_chunk000,8,0.638383,HA applied for asylum in May 2015 alleging that he is registered with UNRWA ...
9,0,Asylum Definition And Assessment,5890_chunk000,9,0.638341,"Six Iraqi nationals, a family of two parents and their four minor children, ..."
